# The DNN malware detector

The central detector of the whole project: a multilayer perceptron over **10,000 binary static features**, trained to separate benign from malicious Android apps. Almost every attack and defence in this repository is measured against this model.

The notebook is organised in two parts that share one set of helpers:

- **Part A** trains and evaluates the detector on the redefined feature matrix.
- **Part B** repeats the build on **Drebin** features and studies the **class-imbalance problem** — the training data is heavily imbalanced, and three ways of handling it are trained and compared.

*Consolidated from `2_DNN_model_2.ipynb` and `2_DNN_model_drebin.ipynb` (originally in `mal_adv3`). Shared helpers (imports, the weight sampler, the model class, the learning-rate schedule, evaluation and the PR/ROC code) appear once, in Step 2; the Drebin part reuses them. Cell outputs were cleared — run the notebook to regenerate them. Data files download automatically from the original `mal_adv3` repository in Step 3.*

## Step 1 — Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import matplotlib.pyplot as plt

torch.manual_seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Step 2 — Shared helpers

Everything both datasets use: the class-weight sampler for imbalanced batches, the model definition, the learning-rate schedule and the evaluation routine. In the original Drebin notebook the learning-rate schedule decayed at epochs (30, 40, 45) instead of (20, 30, 40); the difference is noted where it matters in Part B.

In [ ]:
def weight_sampler(labels_train):

    # count occurance of each class
    labels_train = labels_train.to(torch.int8)
    unique, counts = np.unique(labels_train.to(torch.int8), return_counts=True)

    # calcuate weight of each class
    class_weights = [1.0/c for c in counts]

    # assign weight to each sample
    sample_weights = [class_weights[i] for i in labels_train]

    # Create WeightedRandomSampler
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    return sampler

In [ ]:
class MalwareDetectionModel(nn.Module):
    def __init__(self, input_size=10000, hidden_1_size=200, hidden_2_size=200, num_labels=2, dropout_prob=0.6):
        super(MalwareDetectionModel, self).__init__()

        self.input_size = input_size
        self.hidden_1_size = hidden_1_size
        self.hidden_2_size = hidden_2_size
        self.num_labels = num_labels
        self.dropout_prob = dropout_prob

        self.fc1 = nn.Linear(input_size, hidden_1_size)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_prob)
        self.fc2 = nn.Linear(hidden_1_size, hidden_2_size)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_prob)
        self.fc3 = nn.Linear(hidden_2_size, num_labels)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = self.log_softmax(x)
        return x


In [ ]:
def adjust_learning_rate(optimizer, epoch, lr_step=(20,30,40), lr_decay_ratio=0.2):
    """Adjust the learning rate based on the epoch number."""
    if epoch == 0:
        optimizer.param_groups[0]['lr'] /= 8
    elif epoch in [1, 2, 3]:  # in step five , we finish warm up ,and start normal learning rate
        optimizer.param_groups[0]['lr'] *= 2
    if epoch in lr_step: # in these steps , we are geting close to optimal point so we need to have shorter step
        optimizer.param_groups[0]['lr'] *= lr_decay_ratio
    return optimizer


In [ ]:
def model_evaluation(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    predictions = []
    true_labels = []

    with torch.no_grad():
        for X_batch, labels_batch in test_loader:
            X_batch, labels_batch = X_batch.to(torch.float32).to(device), labels_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.topk(outputs, k=1)
            predictions.extend(predicted.tolist())
            true_labels.extend(labels_batch.tolist())

    # Convert predictions and true labels to numpy arrays
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)

    # Calculate and print test accuracy
    accuracy = accuracy_score(true_labels, predictions)
    print(f'Test Accuracy: {accuracy:.4f}')

    # Calculate and print precision, recall, and F1-score
    precision = precision_score(true_labels, predictions)
    recall = recall_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions)

    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')

    # Calculate and print true positives, true negatives, false positives, and false negatives
    TP = ((predictions == 1) & (true_labels == 1)).sum()
    TN = ((predictions == 0) & (true_labels == 0)).sum()
    FP = ((predictions == 1) & (true_labels == 0)).sum()
    FN = ((predictions == 0) & (true_labels == 1)).sum()

    print(f'True Positives (TP): {TP}')
    print(f'True Negatives (TN): {TN}')
    print(f'False Positives (FP): {FP}')
    print(f'False Negatives (FN): {FN}')

    # Calculate and print False Negative Rate (FNR) and False Positive Rate (FPR)
    FNR = FN / (FN + TP)
    FPR = FP / (FP + TN)

    print(f'False Negative Rate (FNR): {FNR:.4f}')
    print(f'False Positive Rate (FPR): {FPR:.4f}')


In [ ]:
def plot_function(x1,x2):
    width =0.3
    fig = plt.figure()
    ax = fig.add_axes([0,0,1,1])
    plt.bar(np.arange(1, len(x1)+1), x1, width=width)
    plt.bar(np.arange(1, len(x2)+1)+ width, x2, width=width)
    ax.legend(labels=['Class 0', 'Class 1'])
    plt.show()

## Part A — The redefined feature matrix

### Step 3 — Download and load the data

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/data/X_redefined_sparse_matrix.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/data/labels.pt',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/data/vocab.pkl',
]

In [ ]:
output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)


In [ ]:
# Load dataset
X_redefined = sparse.load_npz("X_redefined_sparse_matrix.npz")
labels_tensor = torch.load('labels.pt')

# Split data into train, validation, and test sets with stratified sampling
X_train_val, X_test, labels_train_val, labels_test = train_test_split(X_redefined, labels_tensor, test_size=0.2, stratify=labels_tensor, random_state=42)
X_train, X_val, labels_train, labels_val = train_test_split(X_train_val, labels_train_val, test_size=0.2, stratify=labels_train_val, random_state=42)

# number of benigns and malicious sample in training dataset
n_ben = (labels_train.squeeze()== 0).sum().item()
n_mal = (labels_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(torch.tensor(X_train.toarray(), dtype=torch.int8), labels_train)
val_dataset = TensorDataset(torch.tensor(X_val.toarray(), dtype=torch.int8), labels_val)
test_dataset = TensorDataset(torch.tensor(X_test.toarray(), dtype=torch.int8), labels_test)

# Define the DataLoader for training, validation, and test sets
batch_size = 256
sampler = weight_sampler(labels_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Clear unnecessary variables
#del X_redefined, labels_tensor, X_train_val, X_test, labels_train_val, labels_test, X_train, X_val, labels_train, labels_val

### Step 4 — Check the class balance in the batches

In [ ]:
# iterate through dataset and plot class distribution in each batch
zeros = []
ones = []
for idx, (x,y) in enumerate(train_loader):
    unique, counts = np.unique(y.numpy(), return_counts=True)
    zeros.append(counts[0])
    ones.append(counts[1])

print('total number of label ones',sum(ones))
print('total number of label zeros',sum(zeros))
plot_function(zeros,ones)

In [ ]:
benign_set = set()
malicious_set = set()

proportions_benign = []
proportions_malicious = []


for epoch in range(10):
    for batch_indices in train_loader.batch_sampler:
        for idx in batch_indices:
            label = train_dataset[idx][1].item()
            if label == 0:
                benign_set.add(idx)
            else:
                malicious_set.add(idx)

    # Calculate proportions
    proportion_benign = len(benign_set) / n_ben
    proportion_malicious = len(malicious_set) / n_mal

    proportions_benign.append(proportion_benign)
    proportions_malicious.append(proportion_malicious)

    print('-----------------------------------------')
    print(f'Epoch {epoch+1}')
    print(f'Proportion of seen benign samples: {proportion_benign:.4f}')
    print(f'Proportion of seen malicious samples: {proportion_malicious:.4f}')

# Plot
epochs = range(1, 11)
plt.plot(epochs, proportions_benign, label='Benign Samples')
plt.plot(epochs, proportions_malicious, label='Malicious Samples')

# Add labels and legend
plt.xlabel('Epochs')
plt.ylabel('Proportion')
plt.title('Proportion of Seen Samples over Epochs')
plt.legend()

# Show plot
plt.show()

### Step 5 — Train the detector

In [ ]:
model = MalwareDetectionModel()
model = model.to(device)
model

In [ ]:
learning_rate = 0.001
weight_decay = 0.
num_epochs = 50
dropout = 0.6

# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

best_f1_score = 0.
best_epoch = 0
total_time = 0.
nbatches = len(train_loader)

for i in range(num_epochs):
    model.train()
    losses, f1_scores = [], []
    optimizer = adjust_learning_rate(optimizer, i)

    for idx_batch, (x_train, y_train) in enumerate(train_loader):
        x_train, y_train = x_train.to(torch.float32).to(device), y_train.to(device)
        optimizer.zero_grad()
        outputs = model(x_train)
        loss_train = criterion(outputs, y_train.view(-1).long())
        loss_train.backward()
        optimizer.step()

        predicted = (outputs.argmax(1)).unsqueeze(1)
        f1_train = f1_score(y_train.cpu().numpy(), predicted.cpu().numpy())

        losses.append(loss_train.item())
        f1_scores.append(f1_train)

    model.eval()
    avg_f1_val = []

    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            outputs = model(x_val)
            predicted = (outputs.argmax(1)).unsqueeze(1)
            f1_val = f1_score(y_val.cpu().numpy(), predicted.cpu().numpy())
            avg_f1_val.append(f1_val)

    avg_f1_val = np.mean(avg_f1_val)

    if avg_f1_val >= best_f1_score:
        best_f1_score = avg_f1_val
        best_epoch = i
        torch.save(model.state_dict(), 'model_DNN.pth')

    print(f'Epoch: {i+1}/{num_epochs} | Training loss: {np.mean(losses):.4f} | Validation F1-score: {avg_f1_val:.4f} | The best validation F1-score: {best_f1_score * 100:.2f} at epoch: {best_epoch+1}')

print(f'Best F1-score: {best_f1_score:.4f} | Best Epoch: {best_epoch+1}')


### Step 6 — Load the trained parameters and evaluate

In [ ]:
# Load model parameters
model_DNN = MalwareDetectionModel().to(device)
model_DNN.load_state_dict(torch.load('model_DNN.pth', map_location=torch.device(device)))

In [ ]:
model_evaluation(model_DNN,test_loader,device=device)

In [ ]:
#here , we define prediction, as probability(not binary)

# Evaluate the trained model on the test set
model.eval()  # Set the model to evaluation mode
predictions = []
true_labels = []

with torch.no_grad():
    for X_batch, labels_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.topk(outputs, k=1)
        predictions.extend(predicted.tolist())
        true_labels.extend(labels_batch.tolist())

# Convert predictions and true labels to numpy arrays
predictions = np.array(predictions)
true_labels = np.array(true_labels)


# Calculate precision and recall
precision, recall, thresholds = precision_recall_curve(true_labels, predictions)

# Calculate F1-score for each threshold
f1_scores = 2 * (precision * recall) / (precision + recall)

# Find the threshold that maximizes the F1-score
best_f1_threshold = thresholds[np.argmax(f1_scores)]
best_f1_score = np.max(f1_scores)

print(f'Best F1-score: {best_f1_score:.4f} at threshold: {best_f1_threshold:.4f}')


### Step 7 — Precision–recall and ROC curves

In [ ]:

# Calculate precision and recall
precision, recall, _ = precision_recall_curve(true_labels, predictions)

# Plot precision-recall curve
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True)
plt.show()


In [ ]:
# Compute fpr, tpr, thresholds
fpr, tpr, thresholds = roc_curve(true_labels, predictions)

# Compute AUC (Area Under the Curve)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.4f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid()
plt.show()

## Part B — Drebin features and the imbalance study

### Step 8 — Download and load the Drebin matrices

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz'
]

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)


### Step 9 — How imbalanced is Drebin here?

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# Define the DataLoader for training, validation, and test sets
batch_size = 128
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Clear unnecessary variables
#del X_redefined, labels_tensor, X_train_val, X_test, labels_train_val, labels_test, X_train, X_val, labels_train, labels_val

In [ ]:
n_mal/(n_mal+n_ben)

### Step 10 — Train on Drebin

In [ ]:
learning_rate = 0.001
weight_decay = 0.
num_epochs = 50
dropout = 0.6

# Assuming positive class (malware) is label 1
class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class

# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

best_balanced_acc = 0.
best_epoch = 0

for i in range(num_epochs):
    model.train()
    losses = []
    optimizer = adjust_learning_rate(optimizer, i)

    for idx_batch, (x_train, y_train) in enumerate(train_loader):
        x_train, y_train = x_train.to(torch.float32).to(device), y_train.to(device)
        optimizer.zero_grad()
        outputs = model(x_train)
        loss_train = criterion(outputs, y_train.view(-1).long())
        loss_train.backward()
        optimizer.step()
        losses.append(loss_train.item())

    model.eval()
    balanced_acc_values = []

    with torch.no_grad():
        model_evaluation(model, val_loader, device)
        torch.save(model.state_dict(), f'model_DNN_drebin_imbalanced_epoch_{i+1}.pth')  # Save the model after each epoch

    print(f'Epoch: {i+1}/{num_epochs} | Training loss: {np.mean(losses):.4f}')
    print('----------------------------------------------------------')


In [ ]:
# Load model parameters
for i in range(30):
  print('------------------------------------------------')
  print(f'model_DNN_drebin_imbalanced_epoch_{i+1}.pth')
  model_DNN_drebin_imbalanced = MalwareDetectionModel().to(device)
  model_DNN_drebin_imbalanced.load_state_dict(torch.load(f'model_DNN_drebin_imbalanced_epoch_{i+1}.pth', map_location=torch.device(device)))
  model_evaluation(model_DNN_drebin_imbalanced,test_loader,device=device)

Three runs follow, differing only in how they score/handle the imbalance: plain training judged by **FNR**, training judged by **balanced accuracy**, and training with **class-balanced weights**. The original notebook's own note on balanced accuracy is kept before the runs.

balanced_accuracy_score however works differently in that it returns the average accuracy per class, which is a different metric. Say your 1000 labels are from 2 classes with 750 observations in class 1 and 250 in class 2. If you miss-predict 10 in each class, you have an accuracy of 740/750= 98.7% in class 1 and 240/250=96% in class 2. balanced_accuracy_score would then return (98.7%+96%)/2 = 97.35%. So I believe the program to work as expected, based on the documentation.

In [ ]:
import torch
from sklearn.metrics import balanced_accuracy_score

learning_rate = 0.001
weight_decay = 0.
num_epochs = 50
dropout = 0.6

# Assuming positive class (malware) is label 1
class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class

# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

best_balanced_acc = 0.
best_epoch = 0

for i in range(num_epochs):
    model.train()
    losses = []
    #optimizer = adjust_learning_rate(optimizer, i)

    for idx_batch, (x_train, y_train) in enumerate(train_loader):
        x_train, y_train = x_train.to(torch.float32).to(device), y_train.to(device)
        optimizer.zero_grad()
        outputs = model(x_train)
        loss_train = criterion(outputs, y_train.view(-1).long())
        loss_train.backward()
        optimizer.step()
        losses.append(loss_train.item())

    model.eval()
    balanced_acc_values = []

    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            outputs = model(x_val)
            predicted = outputs.argmax(1)
            balanced_acc = balanced_accuracy_score(y_val.cpu().numpy(), predicted.cpu().numpy())
            balanced_acc_values.append(balanced_acc)

    avg_balanced_acc = np.mean(balanced_acc_values)

    if avg_balanced_acc >= best_balanced_acc:
        best_balanced_acc = avg_balanced_acc
        best_epoch = i
        torch.save(model.state_dict(), 'model_DNN_drebin_imbalanced_balanced_acc.pth')

    print(f'Epoch: {i+1}/{num_epochs} | Training loss: {np.mean(losses):.4f} | Validation Balanced Accuracy: {avg_balanced_acc:.4f} | Best Validation Balanced Accuracy: {best_balanced_acc:.4f} at epoch: {best_epoch+1}')

print(f'Best Balanced Accuracy: {best_balanced_acc:.4f} | Best Epoch: {best_epoch+1}')


In [ ]:
import torch
from sklearn.metrics import confusion_matrix

learning_rate = 0.001
weight_decay = 0.
num_epochs = 50
dropout = 0.6

# Assuming positive class (malware) is label 1
class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class

# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

best_fnr = float('inf')
best_epoch = 0

for i in range(num_epochs):
    model.train()
    losses = []
    optimizer = adjust_learning_rate(optimizer, i)

    for idx_batch, (x_train, y_train) in enumerate(train_loader):
        x_train, y_train = x_train.to(torch.float32).to(device), y_train.to(device)
        optimizer.zero_grad()
        outputs = model(x_train)
        loss_train = criterion(outputs, y_train.view(-1).long())
        loss_train.backward()
        optimizer.step()
        losses.append(loss_train.item())

    model.eval()
    fnr_values = []

    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            outputs = model(x_val)
            predicted = outputs.argmax(1)
            conf_matrix = confusion_matrix(y_val.cpu().numpy(), predicted.cpu().numpy())
            fnr = conf_matrix[1, 0] / (conf_matrix[1, 0] + conf_matrix[1, 1])  # False Negative Rate
            fnr_values.append(fnr)

    avg_fnr = np.mean(fnr_values)

    if avg_fnr <= best_fnr:
        best_fnr = avg_fnr
        best_epoch = i
        torch.save(model.state_dict(), 'model_DNN_drebin_imbalanced_fnr.pth')

    print(f'Epoch: {i+1}/{num_epochs} | Training loss: {np.mean(losses):.4f} | Validation FNR: {avg_fnr:.4f} | Best Validation FNR: {best_fnr:.4f} at epoch: {best_epoch+1}')

print(f'Best FNR: {best_fnr:.4f} | Best Epoch: {best_epoch+1}')


In [ ]:
learning_rate = 0.001
weight_decay = 0.
num_epochs = 50
dropout = 0.6

# Assuming positive class (malware) is label 1
class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class


# Define Loss Function and Optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

best_f1_score = 0.
best_epoch = 0
total_time = 0.
nbatches = len(train_loader)

for i in range(num_epochs):
    model.train()
    losses, f1_scores = [], []
    optimizer = adjust_learning_rate(optimizer, i)

    for idx_batch, (x_train, y_train) in enumerate(train_loader):
        x_train, y_train = x_train.to(torch.float32).to(device), y_train.to(device)
        optimizer.zero_grad()
        outputs = model(x_train)
        loss_train = criterion(outputs, y_train.view(-1).long())
        loss_train.backward()
        optimizer.step()

        predicted = (outputs.argmax(1)).unsqueeze(1)
        f1_train = f1_score(y_train.cpu().numpy(), predicted.cpu().numpy(), zero_division=1)

        losses.append(loss_train.item())
        f1_scores.append(f1_train)

    model.eval()
    avg_f1_val = []

    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            outputs = model(x_val)
            predicted = (outputs.argmax(1)).unsqueeze(1)
            f1_val = f1_score(y_val.cpu().numpy(), predicted.cpu().numpy())
            avg_f1_val.append(f1_val)

    avg_f1_val = np.mean(avg_f1_val)

    if avg_f1_val >= best_f1_score:
        best_f1_score = avg_f1_val
        best_epoch = i
        torch.save(model.state_dict(), 'model_DNN_drebin_balancedWeight.pth')

    print(f'Epoch: {i+1}/{num_epochs} | Training loss: {np.mean(losses):.4f} | Validation F1-score: {avg_f1_val:.4f} | The best validation F1-score: {best_f1_score * 100:.2f} at epoch: {best_epoch+1}')

print(f'Best F1-score: {best_f1_score:.4f} | Best Epoch: {best_epoch+1}')


### Step 11 — Load the three variants and compare them

In [ ]:
# Load model parameters
model_DNN_drebin_imbalanced_fnr = MalwareDetectionModel().to(device)
model_DNN_drebin_imbalanced_fnr.load_state_dict(torch.load('model_DNN_drebin_imbalanced_fnr.pth', map_location=torch.device(device)))

In [ ]:
# Load model parameters
model_DNN_drebin_imbalanced_balanced_acc = MalwareDetectionModel().to(device)
model_DNN_drebin_imbalanced_balanced_acc.load_state_dict(torch.load('model_DNN_drebin_imbalanced_balanced_acc.pth', map_location=torch.device(device)))

In [ ]:
# Load model parameters
model_DNN_drebin_balancedWeight = MalwareDetectionModel().to(device)
model_DNN_drebin_balancedWeight.load_state_dict(torch.load('model_DNN_drebin_balancedWeight.pth', map_location=torch.device(device)))

In [ ]:
model_evaluation(model_DNN_drebin_imbalanced_fnr,test_loader,device=device)

In [ ]:
model_evaluation(model_DNN_drebin_balancedWeight,test_loader,device=device)

In [ ]:
model_evaluation(model_DNN_drebin_imbalanced_balanced_acc,test_loader,device=device)

### Step 12 — Final curves

The prediction, precision–recall and ROC cells are identical to Step 7 and are not repeated; run those cells on the Drebin model's predictions to get the Drebin curves.